# Cruise Fuel Prediction using XGBoost

Goal:
Predict fuel consumption during the cruise phase using ADS-B derived flight phase features.

### Imports

In [1]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import numpy as np
import pandas as pd
import sqlite3




ModuleNotFoundError: No module named 'sklearn'

### Configuration



In [ ]:
DB_PATH = "../opensky.sqlite"
FEATURE_TABLE = "flight_phase_features"

YEAR = 2022
TARGET_FUEL = "cruise_fuel"
GCD_UNITS = "m" 

USE_PER_METER_TARGET = True
EPS = 1e-6

### Helper Functions

We define helper utilities for:

- Parsing timestamps from flight_id
- Error metrics (MAE, MAPE, WAPE, SMAPE)
- Unit normalization

In [ ]:
def add_datetime_from_flight_id(df: pd.DataFrame, year: int) -> pd.DataFrame:
    parts = df["flight_id"].astype(str).str.split("-", expand=True)
    if parts.shape[1] < 6:
        raise ValueError("flight_id must be: icao24-callsign-mm-dd-hh-mm")

    month = parts[2].astype(int)
    day = parts[3].astype(int)
    hour = parts[4].astype(int)
    minute = parts[5].astype(int)

    df = df.copy()
    df["flight_dt"] = pd.to_datetime(
        {"year": year, "month": month, "day": day, "hour": hour, "minute": minute},
        errors="coerce"
    )
    return df


def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.clip(np.abs(y_true), 1e-6, None)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100


def wape(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    denom = np.sum(np.abs(y_true))
    if denom < 1e-9:
        return np.nan
    return 100 * np.sum(np.abs(y_true - y_pred)) / denom


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    denom = np.clip(np.abs(y_true) + np.abs(y_pred), 1e-6, None)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)


def gcd_to_meters(x):
    if x is None:
        return np.nan
    x = float(x)
    if not np.isfinite(x):
        return np.nan
    return x * 1000.0 if GCD_UNITS.lower() == "km" else x

### Load Feature Table

The feature table contains one row per flight with:
- Cruise phase features (altitude, ground speed, turning, etc.)
- Fuel labels (e.g., cruise_fuel)

In [ ]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql(f'SELECT * FROM "{FEATURE_TABLE}"', conn)
conn.close()

required = ["flight_id", "icao24", "time_s_cruise", "dist_m_cruise", TARGET_FUEL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in feature table: {missing}")

df = add_datetime_from_flight_id(df, YEAR)
df = df[df["flight_dt"].notna()].copy()
df["month"] = df["flight_dt"].dt.month

print("Total rows:", len(df))
df.head(3)

Total rows: 14192


,flight_id,icao24,callsign,aircraft_type_icao_code,estdepartureairport,estarrivalairport,great_circle_distance,time_s_takeoff,dist_m_takeoff,alt_gain_m_takeoff,...,mean_gs_mps_cruise,p95_gs_mps_cruise,std_gs_mps_cruise,mean_vr_mps_cruise,p95_abs_vr_mps_cruise,turn_sum_deg_cruise,n_points_cruise,total_fuel_modeled,flight_dt,month
0,4784c2-NOZ132-08-19-13-45,4784c2,NOZ132,B738,ENBR,ENHD,159630.0,60.0,5198.847094,510.54,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,944.66,2022-08-19 13:45:00,8
1,4784c2-NOZ132-10-10-13-51,4784c2,NOZ132,B738,ENBR,ENZV,159630.0,40.0,3616.344493,518.16,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,864.98,2022-10-10 13:51:00,10
2,4784c2-NOZ132-10-12-13-57,4784c2,NOZ132,B738,ENBR,ENZV,159630.0,50.0,4121.323715,556.26,...,167.753352,174.109385,9.987541,-0.65024,1.235456,0.289847,2.0,946.38,2022-10-12 13:57:00,10


### Cruise filtering + Feature Engineering

We keep cruise flights only and define:
- `has_cruise`
- log-distance and interaction terms
- normalized great-circle distance (meters)

In [ ]:
df["has_cruise"] = (df["time_s_cruise"].fillna(0.0) > 0.0).astype(int)

df["log_dist_m_cruise"] = np.log1p(df["dist_m_cruise"].fillna(0.0))
df["alt_x_logdist"] = df["mean_alt_m_cruise"].fillna(0.0) * df["log_dist_m_cruise"]
df["gs_x_logdist"]  = df["mean_gs_mps_cruise"].fillna(0.0) * df["log_dist_m_cruise"]

if "great_circle_distance" in df.columns:
    df["great_circle_distance_m"] = df["great_circle_distance"].apply(gcd_to_meters)
else:
    df["great_circle_distance_m"] = np.nan

### Temporal Data Split

We split the dataset into:

- Development period: January–October  
- Test period: November–December  

The development period is used for time-based cross-validation and hyperparameter selection.

The test period remains completely unseen until final evaluation to ensure an unbiased estimate of model performance.

In [ ]:
dev  = df[df["month"] <= 10].copy()   # Jan–Oct
test = df[df["month"] >= 11].copy()   # Nov–Dec

print("Development:", len(dev), "| Test:", len(test))

Development: 10686 | Test: 3506


### Cruise-only training subset and rate target

We train only where:
- cruise exists
- cruise distance > 0
- cruise_fuel is known

Target is cruise fuel per meter:
y_rate = cruise_fuel / dist_m_cruise

Optionally we apply log1p to stabilize the distribution.

In [ ]:
def cruise_subset(dfx: pd.DataFrame) -> pd.DataFrame:
    return dfx[
        (dfx["dist_m_cruise"].fillna(0.0) > 0.0) &
        (dfx[TARGET_FUEL].notna()) &
        np.isfinite(dfx[TARGET_FUEL])
    ].copy()

dev_cruise  = cruise_subset(dev)
test_cruise = cruise_subset(test)

print("Cruise development:", len(dev_cruise))
print("Cruise test:", len(test_cruise))

# rate target (strictly positive, for reg:gamma)
dev_cruise["y_rate"] = dev_cruise[TARGET_FUEL] / np.clip(dev_cruise["dist_m_cruise"], EPS, None)
dev_cruise["y_rate"] = np.clip(dev_cruise["y_rate"], EPS, None)

dev_cruise["y_rate"] = dev_cruise[TARGET_FUEL] / np.clip(dev_cruise["dist_m_cruise"], EPS, None)
dev_cruise["y_rate"] = np.clip(dev_cruise["y_rate"], EPS, None)

dev_cruise["y_lograte"] = np.log(dev_cruise["y_rate"])   # log, ikke log1p




Cruise development: 8849
Cruise test: 3122


### Feature Set

We use cruise-specific features + a few route-level features.
The feature list is intentionally explicit to support reproducibility.

In [ ]:
FEATURES = [
    "dist_m_cruise",
    "mean_alt_m_cruise",
    "max_alt_m_cruise",
    "mean_gs_mps_cruise",
    "p95_gs_mps_cruise",
    "std_gs_mps_cruise",
    "mean_vr_mps_cruise",
    "p95_abs_vr_mps_cruise",
    "turn_sum_deg_cruise",
    "great_circle_distance_m",
    "log_dist_m_cruise",
    "alt_x_logdist",
    "gs_x_logdist",
]

FEATURES = [c for c in FEATURES if c in dev_cruise.columns]
print("Features used:", FEATURES)

def make_X(dfx):
    return dfx[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)

Features used: ['dist_m_cruise', 'mean_alt_m_cruise', 'max_alt_m_cruise', 'mean_gs_mps_cruise', 'p95_gs_mps_cruise', 'std_gs_mps_cruise', 'mean_vr_mps_cruise', 'p95_abs_vr_mps_cruise', 'turn_sum_deg_cruise', 'great_circle_distance_m', 'log_dist_m_cruise', 'alt_x_logdist', 'gs_x_logdist']


### Walk-forward CV inside Train

This mirrors the climb notebook: time-ordered folds with inner grouped early stopping.
We evaluate in terms of fuel by converting predicted rate back to fuel.

In [ ]:
K = 5

df_cv = dev_cruise.sort_values("flight_dt").reset_index(drop=True).copy()
df_cv["time_fold"] = pd.qcut(df_cv["flight_dt"].rank(method="first"), q=K, labels=False)

fold_mae, best_iters = [], []

for fold in range(1, K):
    train_fold = df_cv[df_cv["time_fold"] < fold].copy()
    test_fold  = df_cv[df_cv["time_fold"] == fold].copy()

    gss = GroupShuffleSplit(test_size=0.2, random_state=42)
    tr_idx, va_idx = next(gss.split(train_fold, groups=train_fold["icao24"]))

    tr = train_fold.iloc[tr_idx]
    va = train_fold.iloc[va_idx]

    # Features
    X_tr = make_X(tr)
    X_va = make_X(va)
    X_te = make_X(test_fold)

    X_tr_np = np.asarray(X_tr, dtype=np.float32)
    X_va_np = np.asarray(X_va, dtype=np.float32)
    X_te_np = np.asarray(X_te, dtype=np.float32)

    # Targets (log-rate)
    y_tr = tr["y_lograte"].to_numpy(dtype=float)
    y_va = va["y_lograte"].to_numpy(dtype=float)

    fuel_true = test_fold[TARGET_FUEL].to_numpy(dtype=float)
    dist      = test_fold["dist_m_cruise"].to_numpy(dtype=float)

    model = XGBRegressor(
        n_estimators=12000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        reg_alpha=0.0,
        min_child_weight=10,
        gamma=0.0,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=200,
        eval_metric="mae",
        objective="reg:squarederror",
    )

    model.fit(X_tr_np, y_tr, eval_set=[(X_va_np, y_va)], verbose=False)

    pred_log  = model.predict(X_te_np)
    pred_rate = np.exp(pred_log)          # back to rate (>0)
    pred_fuel = pred_rate * dist

    mask_ok = np.isfinite(fuel_true) & np.isfinite(pred_fuel)
    n_ok = int(mask_ok.sum())
    if n_ok == 0:
        print(f"Fold {fold}/{K} | 0 valid samples after filtering -> skipped")
        continue

    mae = mean_absolute_error(fuel_true[mask_ok], pred_fuel[mask_ok])
    fold_mae.append(mae)

    best_iter = model.best_iteration if model.best_iteration is not None else model.n_estimators
    best_iters.append(best_iter)

    print(f"Fold {fold}/{K} | MAE={mae:.2f} | best_iter={best_iter}")

print("\nCV summary")
print(f"Mean MAE : {np.mean(fold_mae):.2f} ± {np.std(fold_mae):.2f}")
avg_best_iter = int(np.mean(best_iters)) if best_iters else 12000
print(f"Avg best_iter: {avg_best_iter}")

Fold 1/5 | MAE=32.03 | best_iter=600
Fold 2/5 | MAE=59.46 | best_iter=644
Fold 3/5 | MAE=30.66 | best_iter=791
Fold 4/5 | MAE=30.47 | best_iter=1062

CV summary
Mean MAE : 38.16 ± 12.32
Avg best_iter: 774


### Final model 

After cross-validation, we train the final cruise model on the entire development period (January–October) using the selected number of boosting rounds.

The model predicts cruise fuel rate, which is then converted to total cruise fuel by multiplying by cruise distance:

fuel = rate × distance

We evaluate the final model on the unseen test period (November–December) using MAE, WAPE, and SMAPE.

In [ ]:
# ===== FINAL TRAIN (matches CV: log-rate) =====
X_dev_np = np.asarray(make_X(dev_cruise), dtype=np.float32)
y_dev_log = dev_cruise["y_lograte"].to_numpy(dtype=float)

final_cruise_model = XGBRegressor(
    n_estimators=avg_best_iter,
    learning_rate=0.03,   # must match CV
    max_depth=4,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=10,
    gamma=0.0,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror",
)

final_cruise_model.fit(X_dev_np, y_dev_log)

# ===== FINAL TEST =====
X_test_np   = np.asarray(make_X(test_cruise), dtype=np.float32)
y_test_fuel = test_cruise[TARGET_FUEL].to_numpy(dtype=float)
dist_test   = test_cruise["dist_m_cruise"].to_numpy(dtype=float)

pred_lograte  = final_cruise_model.predict(X_test_np)
pred_rate_test = np.exp(pred_lograte)          # <- IMPORTANT (back to rate)
pred_fuel_test = pred_rate_test * dist_test

mask_ok_test = np.isfinite(y_test_fuel) & np.isfinite(pred_fuel_test)

print("FINAL TEST (Cruise-only)")
print(f"MAE : {mean_absolute_error(y_test_fuel[mask_ok_test], pred_fuel_test[mask_ok_test]):.2f}")
print(f"WAPE: {wape(y_test_fuel[mask_ok_test], pred_fuel_test[mask_ok_test]):.2f}%")
print(f"SMAPE: {smape(y_test_fuel[mask_ok_test], pred_fuel_test[mask_ok_test]):.2f}%")
print(f"MAPE: {mape(y_test_fuel, pred_fuel_test):.2f}%")

FINAL TEST (Cruise-only)
MAE : 31.34
WAPE: 10.86%
SMAPE: 16.04%
MAPE: 18.79%


## Feature importance

In [ ]:
imp = pd.Series(final_cruise_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
imp

great_circle_distance_m    0.137874
p95_gs_mps_cruise          0.125300
log_dist_m_cruise          0.108679
dist_m_cruise              0.092433
max_alt_m_cruise           0.079017
mean_vr_mps_cruise         0.077104
gs_x_logdist               0.068121
p95_abs_vr_mps_cruise      0.064929
turn_sum_deg_cruise        0.054384
mean_gs_mps_cruise         0.052567
mean_alt_m_cruise          0.050529
std_gs_mps_cruise          0.047809
alt_x_logdist              0.041254
dtype: float32

### True vs Predicted descent fuel plot


We also generate a scatter plot comparing true and predicted climb fuel values, including a reference line (y = x).

In [3]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(y_test_fuel, pred_fuel_test, alpha=0.35, s=10)
mx = max(float(np.max(y_test_fuel)), float(np.max(pred_fuel_test)))
plt.plot([0, mx], [0, mx], linewidth=2)
plt.xlabel("True cruise fuel")
plt.ylabel("Predicted cruise fuel")
plt.title("True vs Predicted (Cruise test: Nov–Dec)")
plt.grid(True, alpha=0.3)
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

### Generate csv file with flight id and corresponding fuel prediction

Finally, we export the test-set predictions to a CSV file for downstream analysis.

The output file contains:
- Flight identifier
- Predicted cruise fuel

In [ ]:
from pathlib import Path


csv_folder = Path("csv_files")

file_path = csv_folder / "pred_cruise.csv"
pred_df = pd.DataFrame({
    "flight_id": test_cruise["flight_id"].values,
    "pred_cruise": pred_fuel_test
})
pred_df.to_csv(file_path, index=False)